In [1]:
"""
NarrativeShield — Annotator Statistics Report (FINAL, reviewer-proof version)
==============================================================================
This is a corrected rewrite of the original annotator_statistics.py, after
diagnosing three real issues in the human validation data:

  1. facts_preserved_binary and register_matches_persona are CONSTANT
     (all 1s) across all 3 annotators, all 300 rows, all 3 personas.
     Cohen's kappa / Fleiss' kappa are mathematically UNDEFINED when a
     column has zero variance (pe = 1 -> division by zero). The original
     script silently returned 1.0 in this case, which is misleading.
     This version instead reports these columns as "constant / kappa
     undefined by construction" and gives you the language to state this
     honestly as a limitation (see printed report + methodology note at
     bottom of this file).

  2. McNemar's test had a bug: after computing a safe manual result, the
     code called statsmodels' mcnemar() on a [[0, n01], [n10, 0]] table
     and OVERWROTE the safe result. When n01 == n10 == 0 (no discordant
     pairs at all, as is the case for both binary columns here),
     statsmodels was returning chi2=inf, p=0.0 — i.e. spuriously
     "significant" — for what is actually a completely undefined/
     degenerate test (there is nothing to compare). This version fixes
     that: when n01 + n10 == 0, we report the test as N/A, not
     significant.

  3. Annotator_2 rated narrative_realism_likert as a flat 5 across all
     300 rows (confirmed genuine after checking raw dtypes — not a
     coercion bug). This is a real straightlining pattern that tanks
     Krippendorff's alpha when all 3 annotators are pooled. This version
     reports Krippendorff's alpha / ICC BOTH with all 3 annotators AND
     with Annotator_2 excluded, so you can present both transparently.

Everything else (descriptive stats, Kruskal-Wallis, Mann-Whitney,
Chi-square) is unaffected by these bugs and is carried over, cleaned up,
and given consistent handling for edge cases + clearer output.

REQUIREMENTS
------------
pip install pandas numpy scipy scikit-learn statsmodels krippendorff openpyxl

USAGE
-----
1. Edit FILES, ID_COLUMNS, PERSONA_COLUMN below if your sheets differ.
2. Run: python annotator_statistics_FINAL.py
3. Read the console report. CSVs + a markdown summary land in OUTPUT_DIR,
   ready to paste into an appendix.
"""

import os
import warnings
import numpy as np
import pandas as pd
from itertools import combinations

warnings.filterwarnings("ignore")

# ============================================================
# EDIT THIS SECTION
# ============================================================

FILES = {
    "Annotator_1": "Annotation_B.xlsx",
    "Annotator_2": "Annotation_M.xlsx",
    "Annotator_3": "Annotation_R.xlsx",
}

SHEET_NAME = 0

COL_FACTS = "facts_preserved_binary"
COL_REALISM = "narrative_realism_likert"
COL_REGISTER = "register_matches_persona"

ID_COLUMNS = ["question_id"]
PERSONA_COLUMN = "persona"

# Annotator(s) to exclude in the "sensitivity" pass for realism agreement.
# Based on diagnostics: Annotator_2 rated realism as a flat 5 across all
# 300 items (confirmed genuine, not a data bug), which is a legitimate
# straightlining pattern for reliability purposes.
STRAIGHTLINE_ANNOTATORS = ["Annotator_2"]

OUTPUT_DIR = "./stats_output_final"

# ============================================================
# LOAD + ALIGN
# ============================================================

def load_annotator(name, path):
    df = pd.read_excel(path, sheet_name=SHEET_NAME, dtype=object)
    missing = [c for c in ID_COLUMNS + [PERSONA_COLUMN, COL_FACTS, COL_REALISM, COL_REGISTER]
               if c not in df.columns]
    if missing:
        raise ValueError(f"[{name}] Missing expected column(s): {missing}. "
                          f"Available columns: {list(df.columns)}")

    df = df[ID_COLUMNS + [PERSONA_COLUMN, COL_FACTS, COL_REALISM, COL_REGISTER]].copy()
    df[PERSONA_COLUMN] = df[PERSONA_COLUMN].astype(str).str.strip().str.lower()

    for col in [COL_FACTS, COL_REALISM, COL_REGISTER]:
        coerced = pd.to_numeric(df[col], errors="coerce")
        n_bad = (coerced.isna() & df[col].notna()).sum()
        if n_bad:
            bad_vals = df.loc[coerced.isna() & df[col].notna(), col].unique()
            print(f"  [WARNING] {name}.{col}: {n_bad} value(s) could not be parsed "
                  f"as numeric and became missing: {list(bad_vals)}")
        df[col] = coerced

    df["annotator"] = name
    return df


def load_all():
    frames = {}
    for name, path in FILES.items():
        print(f"Loading {name} <- {path}")
        frames[name] = load_annotator(name, path)
    return frames


def is_constant(series):
    """True if a column has zero variance (only one unique non-NaN value)."""
    return series.dropna().nunique() <= 1


# ============================================================
# AGREEMENT STATISTICS
# ============================================================

def cohens_kappa(a, b):
    """
    Manual Cohen's Kappa. Returns (kappa, is_defined).
    is_defined=False when either rater's column is constant (pe == 1),
    in which case kappa is mathematically undefined -- we do NOT
    silently return 1.0.
    """
    a = np.asarray(a)
    b = np.asarray(b)
    labels = sorted(set(a) | set(b))
    n = len(a)
    idx = {l: i for i, l in enumerate(labels)}
    conf = np.zeros((len(labels), len(labels)))
    for x, y in zip(a, b):
        conf[idx[x], idx[y]] += 1
    po = np.trace(conf) / n
    row_marg = conf.sum(axis=1) / n
    col_marg = conf.sum(axis=0) / n
    pe = np.sum(row_marg * col_marg)
    if np.isclose(pe, 1.0):
        return np.nan, False
    return (po - pe) / (1 - pe), True


def percent_agreement(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    return float(np.mean(a == b))


def fleiss_kappa_manual(rating_matrix):
    """Returns (kappa, is_defined). rating_matrix: (n_items, n_categories) counts."""
    n_items, n_categories = rating_matrix.shape
    n_raters = rating_matrix.sum(axis=1)[0]
    p_j = rating_matrix.sum(axis=0) / (n_items * n_raters)
    P_i = (np.sum(rating_matrix ** 2, axis=1) - n_raters) / (n_raters * (n_raters - 1))
    P_bar = np.mean(P_i)
    P_e_bar = np.sum(p_j ** 2)
    if np.isclose(P_e_bar, 1.0):
        return np.nan, False
    return (P_bar - P_e_bar) / (1 - P_e_bar), True


def try_fleiss_kappa(rating_matrix):
    try:
        from statsmodels.stats.inter_rater import fleiss_kappa
        val = fleiss_kappa(rating_matrix, method="fleiss")
        if np.isnan(val):
            return val, False
        return val, True
    except ImportError:
        return fleiss_kappa_manual(rating_matrix)


def krippendorff_alpha_manual(data, level="ordinal"):
    all_vals = sorted({v for rater in data for v in rater if not (isinstance(v, float) and np.isnan(v))})
    if len(all_vals) <= 1:
        return np.nan
    val_to_rank = {v: i for i, v in enumerate(all_vals)}

    def dist(v1, v2):
        if level == "nominal":
            return 0.0 if v1 == v2 else 1.0
        r1, r2 = val_to_rank[v1], val_to_rank[v2]
        lo, hi = min(r1, r2), max(r1, r2)
        return 0.0 if lo == hi else (hi - lo) ** 2

    n_items = len(data[0])
    pairs = []
    for item_idx in range(n_items):
        vals = [rater[item_idx] for rater in data
                if not (isinstance(rater[item_idx], float) and np.isnan(rater[item_idx]))]
        for i in range(len(vals)):
            for j in range(i + 1, len(vals)):
                pairs.append((vals[i], vals[j]))
    if not pairs:
        return np.nan
    Do = np.mean([dist(v1, v2) for v1, v2 in pairs])
    all_ratings = [v for rater in data for v in rater
                   if not (isinstance(v, float) and np.isnan(v))]
    De_pairs = [(v1, v2) for v1 in all_ratings for v2 in all_ratings]
    De = np.mean([dist(v1, v2) for v1, v2 in De_pairs]) if De_pairs else np.nan
    if De == 0:
        return np.nan  # no variance at all -- undefined, not 1.0
    return 1 - (Do / De)


def try_krippendorff_alpha(rating_matrix_raters_x_items, level="ordinal"):
    try:
        import krippendorff
        return krippendorff.alpha(reliability_data=rating_matrix_raters_x_items, level_of_measurement=level)
    except ImportError:
        return krippendorff_alpha_manual(rating_matrix_raters_x_items, level=level)


def icc_2_1(rating_matrix):
    """
    ICC(2,1) -- two-way random effects, single rater, absolute agreement.
    rating_matrix: (n_items, n_raters). Returns nan if undefined
    (e.g. zero between-item variance).
    """
    n, k = rating_matrix.shape
    mean_items = rating_matrix.mean(axis=1)
    mean_raters = rating_matrix.mean(axis=0)
    grand_mean = rating_matrix.mean()

    SST = np.sum((rating_matrix - grand_mean) ** 2)
    SSR = k * np.sum((mean_items - grand_mean) ** 2)
    SSC = n * np.sum((mean_raters - grand_mean) ** 2)
    SSE = SST - SSR - SSC

    MSR = SSR / (n - 1)
    MSC = SSC / (k - 1) if k > 1 else 0
    MSE = SSE / ((n - 1) * (k - 1)) if k > 1 else np.nan

    denom = MSR + (k - 1) * MSE + (k / n) * (MSC - MSE)
    if denom == 0 or np.isnan(denom):
        return np.nan
    return (MSR - MSE) / denom


def kappa_interpretation(k):
    if k is None or (isinstance(k, float) and np.isnan(k)):
        return "undefined (no variance to assess)"
    if k < 0:
        return "poor (worse than chance)"
    if k < 0.20:
        return "slight"
    if k < 0.40:
        return "fair"
    if k < 0.60:
        return "moderate"
    if k < 0.80:
        return "substantial"
    return "almost perfect"


# ============================================================
# CROSS-PERSONA STATISTICAL TESTS
# ============================================================

def mcnemar_pair(binary_a, binary_b):
    """
    Paired McNemar's test (continuity-corrected).
    Returns (statistic, p_value, (n01, n10), is_defined).
    is_defined=False when n01 + n10 == 0 -- there is nothing discordant
    to test, so the test does not apply (NOT "significant").
    This function is authoritative -- nothing downstream overrides it.
    """
    from scipy.stats import chi2

    a = np.asarray(binary_a)
    b = np.asarray(binary_b)
    n01 = int(np.sum((a == 0) & (b == 1)))
    n10 = int(np.sum((a == 1) & (b == 0)))
    n = n01 + n10
    if n == 0:
        return np.nan, np.nan, (n01, n10), False
    stat = (abs(n01 - n10) - 1) ** 2 / n
    p = 1 - chi2.cdf(stat, df=1)
    return stat, p, (n01, n10), True


def cochrans_q_manual(binary_matrix):
    """binary_matrix: (n_items, n_conditions) of 0/1. Returns (Q, p, is_defined)."""
    from scipy.stats import chi2
    n, k = binary_matrix.shape
    col_sums = binary_matrix.sum(axis=0)
    row_sums = binary_matrix.sum(axis=1)
    grand = binary_matrix.sum()

    denominator = k * grand - np.sum(row_sums ** 2)
    if denominator == 0:
        return np.nan, np.nan, False
    numerator = (k - 1) * (k * np.sum(col_sums ** 2) - grand ** 2)
    Q = numerator / denominator
    p = 1 - chi2.cdf(Q, df=k - 1)
    return Q, p, True


def try_cochrans_q(binary_matrix):
    try:
        from statsmodels.stats.contingency_tables import cochrans_q
        res = cochrans_q(binary_matrix)
        if np.isnan(res.statistic):
            return np.nan, np.nan, False
        return res.statistic, res.pvalue, True
    except (ImportError, Exception):
        return cochrans_q_manual(binary_matrix)


# ============================================================
# MAIN ANALYSIS
# ============================================================

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    frames = load_all()

    annotator_names = list(FILES.keys())
    personas = sorted(set().union(*[set(df[PERSONA_COLUMN]) for df in frames.values()]))
    print(f"\nDetected personas: {personas}")

    merged = None
    for name, df in frames.items():
        renamed = df.rename(columns={
            COL_FACTS: f"{COL_FACTS}__{name}",
            COL_REALISM: f"{COL_REALISM}__{name}",
            COL_REGISTER: f"{COL_REGISTER}__{name}",
        }).drop(columns=["annotator"])
        merged = renamed if merged is None else pd.merge(
            merged, renamed, on=ID_COLUMNS + [PERSONA_COLUMN], how="inner"
        )

    print(f"Aligned rows across all annotators: {len(merged)} "
          f"(expect {len(personas)} personas x N questions)")
    if len(merged) == 0:
        print("[FATAL] No rows aligned across annotators.")
        return
    merged.to_csv(os.path.join(OUTPUT_DIR, "merged_aligned_annotations.csv"), index=False)

    report_lines = []  # for the markdown summary
    report_lines.append("# NarrativeShield — Annotator Validation Statistics\n")

    # ================================================================
    # 1. DESCRIPTIVE STATISTICS
    # ================================================================
    print(f"\n{'='*70}\n1. DESCRIPTIVE STATISTICS\n{'='*70}")
    report_lines.append("## 1. Descriptive Statistics\n")
    desc_rows = []
    for name, df in frames.items():
        for persona in personas:
            sub = df[df[PERSONA_COLUMN] == persona]
            desc_rows.append({
                "annotator": name, "persona": persona, "n": len(sub),
                "facts_preserved_rate": sub[COL_FACTS].mean(),
                "register_match_rate": sub[COL_REGISTER].mean(),
                "realism_mean": sub[COL_REALISM].mean(),
                "realism_median": sub[COL_REALISM].median(),
                "realism_sd": sub[COL_REALISM].std(),
            })
    all_df = pd.concat(frames.values(), ignore_index=True)
    for persona in personas:
        sub = all_df[all_df[PERSONA_COLUMN] == persona]
        desc_rows.append({
            "annotator": "POOLED (all annotators)", "persona": persona, "n": len(sub),
            "facts_preserved_rate": sub[COL_FACTS].mean(),
            "register_match_rate": sub[COL_REGISTER].mean(),
            "realism_mean": sub[COL_REALISM].mean(),
            "realism_median": sub[COL_REALISM].median(),
            "realism_sd": sub[COL_REALISM].std(),
        })
    desc_df = pd.DataFrame(desc_rows)
    print(desc_df.to_string(index=False))
    desc_df.to_csv(os.path.join(OUTPUT_DIR, "descriptive_stats.csv"), index=False)
    report_lines.append(desc_df.to_markdown(index=False) + "\n")

    # Flag constant columns explicitly, up front
    print(f"\n--- Variance check (before computing any agreement stats) ---")
    report_lines.append("## 2. Variance Check\n")
    variance_notes = []
    for col, label in [(COL_FACTS, "Facts Preserved"), (COL_REGISTER, "Register Match")]:
        const_per_annotator = {name: is_constant(df[col]) for name, df in frames.items()}
        all_constant = all(const_per_annotator.values())
        msg = (f"{label}: constant for {sum(const_per_annotator.values())}/{len(annotator_names)} "
               f"annotators -> {'ALL constant (kappa undefined by construction)' if all_constant else 'some variance present'}")
        print(f"  {msg}")
        variance_notes.append(msg)
    report_lines.append("\n".join(f"- {m}" for m in variance_notes) + "\n")

    # ================================================================
    # 2. INTER-ANNOTATOR AGREEMENT
    # ================================================================
    print(f"\n{'='*70}\n3. INTER-ANNOTATOR AGREEMENT\n{'='*70}")
    report_lines.append("## 3. Inter-Annotator Agreement\n")
    agreement_rows = []

    for col, label in [(COL_FACTS, "Facts Preserved"), (COL_REGISTER, "Register Match")]:
        print(f"\n--- {label} (binary) ---")
        report_lines.append(f"### {label} (binary)\n")
        col_is_constant_everywhere = all(is_constant(frames[a][col]) for a in annotator_names)

        if col_is_constant_everywhere:
            note = (f"All annotators rated {label} as constant (single value) across all "
                    f"{len(merged)} items. Cohen's kappa, Fleiss' kappa are mathematically "
                    f"undefined here (p_e = 1, division by zero) -- NOT reported as 1.0. "
                    f"Percent agreement = 1.000 (trivially, since there is no disagreement "
                    f"possible), but this reflects zero variance in the judgment task, not "
                    f"informative reliability. Recommended reporting: state raw agreement "
                    f"rate and explicitly flag that kappa is undefined by construction.")
            print(f"  [NOTE] {note}")
            report_lines.append(f"> **Note:** {note}\n")
            agreement_rows.append({"metric_target": label, "comparison": "all_annotators",
                                    "statistic": "status", "value": "constant_kappa_undefined"})

        pair_kappas = []
        for a, b in combinations(annotator_names, 2):
            va = merged[f"{col}__{a}"]
            vb = merged[f"{col}__{b}"]
            pa = percent_agreement(va, vb)
            kappa, defined = cohens_kappa(va, vb)
            interp = kappa_interpretation(kappa) if defined else "undefined (no variance to assess)"
            print(f"  {a} vs {b}: percent agreement = {pa:.3f}, "
                  f"Cohen's kappa = {'undefined' if not defined else f'{kappa:.3f}'} ({interp})")
            if defined:
                pair_kappas.append(kappa)
            agreement_rows.append({"metric_target": label, "comparison": f"{a} vs {b}",
                                    "statistic": "percent_agreement", "value": pa})
            agreement_rows.append({"metric_target": label, "comparison": f"{a} vs {b}",
                                    "statistic": "cohens_kappa",
                                    "value": kappa if defined else None})

        cat_values = [0, 1]
        rating_matrix = np.zeros((len(merged), len(cat_values)), dtype=int)
        for i, cat in enumerate(cat_values):
            for a in annotator_names:
                rating_matrix[:, i] += (merged[f"{col}__{a}"] == cat).astype(int).values
        fleiss, f_defined = try_fleiss_kappa(rating_matrix)
        print(f"  Fleiss' kappa (all {len(annotator_names)} annotators): "
              f"{'undefined' if not f_defined else f'{fleiss:.3f}'} "
              f"({kappa_interpretation(fleiss) if f_defined else 'undefined (no variance to assess)'})")
        agreement_rows.append({"metric_target": label, "comparison": "all_annotators",
                                "statistic": "fleiss_kappa", "value": fleiss if f_defined else None})
        if pair_kappas:
            print(f"  Mean pairwise Cohen's kappa: {np.mean(pair_kappas):.3f}")
        else:
            print(f"  Mean pairwise Cohen's kappa: undefined (no pair had defined kappa)")

    # --- Realism (ordinal) -- WITH all 3 annotators, and excluding straightliners ---
    print(f"\n--- Narrative Realism (ordinal 1-5) ---")
    report_lines.append("### Narrative Realism (ordinal 1-5)\n")

    def realism_agreement_block(annotator_subset, subset_label):
        print(f"\n  [{subset_label}: {', '.join(annotator_subset)}]")
        report_lines.append(f"**{subset_label}: {', '.join(annotator_subset)}**\n")
        realism_matrix = np.array([merged[f"{COL_REALISM}__{a}"].values for a in annotator_subset], dtype=float)
        alpha = try_krippendorff_alpha(realism_matrix, level="ordinal")
        icc = icc_2_1(realism_matrix.T) if len(annotator_subset) > 1 else np.nan
        print(f"    Krippendorff's alpha (ordinal): {alpha:.3f}" if not np.isnan(alpha) else "    Krippendorff's alpha: undefined")
        print(f"    ICC(2,1) (absolute agreement):  {icc:.3f}" if not np.isnan(icc) else "    ICC(2,1): undefined")
        agreement_rows.append({"metric_target": f"Narrative Realism [{subset_label}]", "comparison": "subset",
                                "statistic": "krippendorff_alpha", "value": alpha})
        agreement_rows.append({"metric_target": f"Narrative Realism [{subset_label}]", "comparison": "subset",
                                "statistic": "ICC(2,1)", "value": icc})
        report_lines.append(f"- Krippendorff's alpha: {alpha:.3f}\n- ICC(2,1): {icc:.3f}\n" if not np.isnan(alpha) else "- Undefined\n")
        return alpha, icc

    realism_agreement_block(annotator_names, "ALL annotators")
    non_straightline = [a for a in annotator_names if a not in STRAIGHTLINE_ANNOTATORS]
    if len(non_straightline) < len(annotator_names):
        realism_agreement_block(non_straightline, "EXCLUDING flagged straightliner(s)")
        print(f"\n  [Flagged straightliner(s): {STRAIGHTLINE_ANNOTATORS} -- rated realism as a "
              f"constant single value across all {len(merged)} items; confirmed genuine on "
              f"manual review, not a data/coercion artifact.]")
        report_lines.append(f"> Flagged straightliner(s): {STRAIGHTLINE_ANNOTATORS} — rated realism "
                             f"as a constant value across all items; confirmed genuine on manual review.\n")

    for a, b in combinations(annotator_names, 2):
        va, vb = merged[f"{COL_REALISM}__{a}"], merged[f"{COL_REALISM}__{b}"]
        pa = percent_agreement(va, vb)
        print(f"  {a} vs {b}: exact percent agreement (Likert) = {pa:.3f}")
        agreement_rows.append({"metric_target": "Narrative Realism", "comparison": f"{a} vs {b}",
                                "statistic": "percent_agreement", "value": pa})

    agreement_df = pd.DataFrame(agreement_rows)
    agreement_df.to_csv(os.path.join(OUTPUT_DIR, "interannotator_agreement.csv"), index=False)

    # ================================================================
    # 3. CROSS-PERSONA COMPARISONS
    # ================================================================
    print(f"\n{'='*70}\n4. CROSS-PERSONA STATISTICAL TESTS\n{'='*70}")
    report_lines.append("## 4. Cross-Persona Statistical Tests\n")
    test_rows = []

    for col, label in [(COL_FACTS, "Facts Preserved"), (COL_REGISTER, "Register Match")]:
        print(f"\n--- McNemar's test: {label}, persona pairs ---")
        report_lines.append(f"### McNemar's test — {label}\n")
        for name in annotator_names:
            df = frames[name]
            wide = df.pivot_table(index=ID_COLUMNS, columns=PERSONA_COLUMN, values=col, aggfunc="first")
            for p1, p2 in combinations(personas, 2):
                if p1 not in wide.columns or p2 not in wide.columns:
                    continue
                sub = wide[[p1, p2]].dropna()
                stat, p, (n01, n10), defined = mcnemar_pair(sub[p1], sub[p2])
                if not defined:
                    print(f"  [{name}] {p1} vs {p2}: N/A -- no discordant pairs "
                          f"(n01={n01}, n10={n10}); test does not apply, NOT significant")
                else:
                    sig = "significant" if p < 0.05 else "n.s."
                    print(f"  [{name}] {p1} vs {p2}: chi2={stat:.3f}, p={p:.4f} ({sig}); n01={n01}, n10={n10}")
                test_rows.append({
                    "test": "McNemar", "target": label, "annotator": name,
                    "comparison": f"{p1} vs {p2}", "statistic": stat if defined else None,
                    "p_value": p if defined else None, "n01": n01, "n10": n10,
                    "defined": defined, "significant_at_0.05": (p < 0.05) if defined else False
                })

        print(f"  [Majority vote across annotators]")
        maj_col = f"{col}_majority"
        merged[maj_col] = merged[[f"{col}__{a}" for a in annotator_names]].mode(axis=1)[0]
        wide_maj = merged.pivot_table(index=ID_COLUMNS, columns=PERSONA_COLUMN, values=maj_col, aggfunc="first")
        for p1, p2 in combinations(personas, 2):
            if p1 not in wide_maj.columns or p2 not in wide_maj.columns:
                continue
            sub = wide_maj[[p1, p2]].dropna()
            stat, p, (n01, n10), defined = mcnemar_pair(sub[p1], sub[p2])
            if not defined:
                print(f"  [majority] {p1} vs {p2}: N/A -- no discordant pairs (n01={n01}, n10={n10})")
            else:
                sig = "significant" if p < 0.05 else "n.s."
                print(f"  [majority] {p1} vs {p2}: chi2={stat:.3f}, p={p:.4f} ({sig}); n01={n01}, n10={n10}")
            test_rows.append({
                "test": "McNemar", "target": label, "annotator": "majority_vote",
                "comparison": f"{p1} vs {p2}", "statistic": stat if defined else None,
                "p_value": p if defined else None, "n01": n01, "n10": n10,
                "defined": defined, "significant_at_0.05": (p < 0.05) if defined else False
            })

        wide_maj_full = wide_maj[personas].dropna()
        if len(personas) > 2 and len(wide_maj_full) > 0:
            Q, pq, q_defined = try_cochrans_q(wide_maj_full.values.astype(int))
            if not q_defined:
                print(f"  [majority] Cochran's Q across all {len(personas)} personas: N/A (no variance)")
            else:
                sig = "significant" if pq < 0.05 else "n.s."
                print(f"  [majority] Cochran's Q across all {len(personas)} personas: Q={Q:.3f}, p={pq:.4f} ({sig})")
            test_rows.append({
                "test": "Cochran's Q", "target": label, "annotator": "majority_vote",
                "comparison": "all personas", "statistic": Q if q_defined else None,
                "p_value": pq if q_defined else None, "n01": None, "n10": None,
                "defined": q_defined, "significant_at_0.05": (pq < 0.05) if q_defined else False
            })

    from scipy.stats import chi2_contingency
    print(f"\n--- Chi-square test of independence (pooled, unpaired cross-check) ---")
    report_lines.append("### Chi-square test of independence (pooled)\n")
    for col, label in [(COL_FACTS, "Facts Preserved"), (COL_REGISTER, "Register Match")]:
        if is_constant(all_df[col]):
            print(f"  {label}: N/A -- column is constant, no contingency to test")
            test_rows.append({"test": "Chi-square independence", "target": label, "annotator": "pooled",
                               "comparison": "all personas", "statistic": None, "p_value": None,
                               "n01": None, "n10": None, "defined": False, "significant_at_0.05": False})
            continue
        contingency = pd.crosstab(all_df[PERSONA_COLUMN], all_df[col])
        chi2_stat, p, dof, _ = chi2_contingency(contingency)
        sig = "significant" if p < 0.05 else "n.s."
        print(f"  {label}: chi2={chi2_stat:.3f}, df={dof}, p={p:.4f} ({sig})")
        test_rows.append({"test": "Chi-square independence", "target": label, "annotator": "pooled",
                           "comparison": "all personas", "statistic": chi2_stat, "p_value": p,
                           "n01": None, "n10": None, "defined": True, "significant_at_0.05": p < 0.05})

    from scipy.stats import kruskal, mannwhitneyu
    print(f"\n--- Kruskal-Wallis H-test: Narrative Realism across personas (pooled) ---")
    report_lines.append("### Kruskal-Wallis — Narrative Realism across personas (pooled)\n")
    groups = [all_df[all_df[PERSONA_COLUMN] == p][COL_REALISM].dropna() for p in personas]
    h_stat, p_kw = kruskal(*groups)
    sig = "significant" if p_kw < 0.05 else "n.s."
    print(f"  H={h_stat:.3f}, p={p_kw:.4f} ({sig})")
    report_lines.append(f"- H={h_stat:.3f}, p={p_kw:.4f} ({sig})\n")
    test_rows.append({"test": "Kruskal-Wallis", "target": "Narrative Realism", "annotator": "pooled",
                       "comparison": "all personas", "statistic": h_stat, "p_value": p_kw,
                       "n01": None, "n10": None, "defined": True, "significant_at_0.05": p_kw < 0.05})

    print(f"  Post-hoc pairwise Mann-Whitney U (Bonferroni-corrected):")
    report_lines.append("**Post-hoc Mann-Whitney U (Bonferroni-corrected):**\n")
    n_pairs = len(list(combinations(personas, 2)))
    for p1, p2 in combinations(personas, 2):
        g1 = all_df[all_df[PERSONA_COLUMN] == p1][COL_REALISM].dropna()
        g2 = all_df[all_df[PERSONA_COLUMN] == p2][COL_REALISM].dropna()
        u_stat, p_mw = mannwhitneyu(g1, g2, alternative="two-sided")
        p_bonf = min(p_mw * n_pairs, 1.0)
        sig = "significant" if p_bonf < 0.05 else "n.s."
        print(f"    {p1} vs {p2}: U={u_stat:.1f}, p={p_mw:.4f}, Bonferroni p={p_bonf:.4f} ({sig})")
        report_lines.append(f"- {p1} vs {p2}: U={u_stat:.1f}, p={p_mw:.4f}, Bonferroni p={p_bonf:.4f} ({sig})\n")
        test_rows.append({"test": "Mann-Whitney U", "target": "Narrative Realism", "annotator": "pooled",
                           "comparison": f"{p1} vs {p2}", "statistic": u_stat, "p_value": p_mw,
                           "n01": None, "n10": None, "defined": True, "significant_at_0.05": p_bonf < 0.05})

    tests_df = pd.DataFrame(test_rows)
    tests_df.to_csv(os.path.join(OUTPUT_DIR, "cross_persona_tests.csv"), index=False)

    with open(os.path.join(OUTPUT_DIR, "summary_report.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(report_lines))

    print(f"\n{'='*70}\nDONE — all outputs written to {OUTPUT_DIR}/\n{'='*70}")
    print("  - descriptive_stats.csv")
    print("  - interannotator_agreement.csv")
    print("  - cross_persona_tests.csv")
    print("  - merged_aligned_annotations.csv")
    print("  - summary_report.md  <- paste sections of this into your appendix")


if __name__ == "__main__":
    main()

Loading Annotator_1 <- Annotation_B.xlsx
Loading Annotator_2 <- Annotation_M.xlsx
Loading Annotator_3 <- Annotation_R.xlsx

Detected personas: ['alpha (pα) — high health literacy', 'beta (pβ) — socioeconomic barrier', 'gamma (pγ) — cultural/somatic register']
Aligned rows across all annotators: 300 (expect 3 personas x N questions)

1. DESCRIPTIVE STATISTICS
              annotator                                persona   n  facts_preserved_rate  register_match_rate  realism_mean  realism_median  realism_sd
            Annotator_1      alpha (pα) — high health literacy 100                   1.0                  1.0      4.680000             5.0    0.601009
            Annotator_1      beta (pβ) — socioeconomic barrier 100                   1.0                  1.0      4.220000             4.0    0.675397
            Annotator_1 gamma (pγ) — cultural/somatic register 100                   1.0                  1.0      3.260000             3.0    0.675995
            Annotator_2      al

In [2]:
"""
Adjacent (+/-1) agreement for Narrative Realism, plus a per-persona
breakdown of disagreement between Annotator_1 and Annotator_3 (the two
non-straightlining annotators).

Run this after annotator_statistics_FINAL.py, or standalone -- it only
needs the three xlsx files.
"""

import pandas as pd
import numpy as np
from itertools import combinations

FILES = {
    "Annotator_1": "Annotation_B.xlsx",
    "Annotator_2": "Annotation_M.xlsx",
    "Annotator_3": "Annotation_R.xlsx",
}

SHEET_NAME = 0
COL_REALISM = "narrative_realism_likert"
ID_COLUMNS = ["question_id"]
PERSONA_COLUMN = "persona"


def load(name, path):
    df = pd.read_excel(path, sheet_name=SHEET_NAME, dtype=object)
    df = df[ID_COLUMNS + [PERSONA_COLUMN, COL_REALISM]].copy()
    df[PERSONA_COLUMN] = df[PERSONA_COLUMN].astype(str).str.strip().str.lower()
    df[COL_REALISM] = pd.to_numeric(df[COL_REALISM], errors="coerce")
    return df


frames = {name: load(name, path) for name, path in FILES.items()}

merged = None
for name, df in frames.items():
    renamed = df.rename(columns={COL_REALISM: f"{COL_REALISM}__{name}"})
    merged = renamed if merged is None else pd.merge(
        merged, renamed, on=ID_COLUMNS + [PERSONA_COLUMN], how="inner"
    )

print(f"Aligned rows: {len(merged)}\n")

annotator_names = list(FILES.keys())

print("="*70)
print("EXACT vs ADJACENT (+/-1) AGREEMENT -- all pairs")
print("="*70)
for a, b in combinations(annotator_names, 2):
    va = merged[f"{COL_REALISM}__{a}"]
    vb = merged[f"{COL_REALISM}__{b}"]
    diff = (va - vb).abs()
    exact = (diff == 0).mean()
    adjacent = (diff <= 1).mean()
    print(f"\n{a} vs {b}:")
    print(f"  Exact agreement:    {exact:.3f} ({exact*100:.1f}%)")
    print(f"  Adjacent (+/-1):    {adjacent:.3f} ({adjacent*100:.1f}%)")
    print(f"  Mean abs difference: {diff.mean():.3f}")
    print(f"  Diff distribution:")
    print(diff.value_counts().sort_index().to_string())

print("\n" + "="*70)
print("PER-PERSONA BREAKDOWN: Annotator_1 vs Annotator_3")
print("(the two non-straightlining annotators)")
print("="*70)
personas = sorted(merged[PERSONA_COLUMN].unique())
for persona in personas:
    sub = merged[merged[PERSONA_COLUMN] == persona]
    va = sub[f"{COL_REALISM}__Annotator_1"]
    vb = sub[f"{COL_REALISM}__Annotator_3"]
    diff = (va - vb).abs()
    exact = (diff == 0).mean()
    adjacent = (diff <= 1).mean()
    print(f"\n{persona}  (n={len(sub)})")
    print(f"  Exact agreement:  {exact:.3f} ({exact*100:.1f}%)")
    print(f"  Adjacent (+/-1):  {adjacent:.3f} ({adjacent*100:.1f}%)")
    print(f"  Mean abs diff:    {diff.mean():.3f}")
    print(f"  A1 mean={va.mean():.2f}, A3 mean={vb.mean():.2f}, "
          f"A1-A3 systematic bias={va.mean()-vb.mean():+.2f}")

print("\n" + "="*70)
print("DIRECTION OF DISAGREEMENT (Annotator_1 vs Annotator_3)")
print("(is one systematically higher, or is it random scatter?)")
print("="*70)
va = merged[f"{COL_REALISM}__Annotator_1"]
vb = merged[f"{COL_REALISM}__Annotator_3"]
signed_diff = va - vb
print(f"A1 > A3: {(signed_diff > 0).sum()} items ({(signed_diff > 0).mean()*100:.1f}%)")
print(f"A1 = A3: {(signed_diff == 0).sum()} items ({(signed_diff == 0).mean()*100:.1f}%)")
print(f"A1 < A3: {(signed_diff < 0).sum()} items ({(signed_diff < 0).mean()*100:.1f}%)")
print(f"Mean signed difference (A1 - A3): {signed_diff.mean():+.3f}")

# Wilcoxon signed-rank test: is the direction of disagreement systematic
# (one annotator consistently higher) or symmetric (random scatter)?
from scipy.stats import wilcoxon
nonzero = signed_diff[signed_diff != 0]
if len(nonzero) > 0:
    stat, p = wilcoxon(nonzero)
    print(f"\nWilcoxon signed-rank test (systematic bias vs symmetric scatter):")
    print(f"  statistic={stat:.3f}, p={p:.4f} "
          f"({'systematic bias' if p < 0.05 else 'no systematic bias -- scatter is symmetric'})")

Aligned rows: 300

EXACT vs ADJACENT (+/-1) AGREEMENT -- all pairs

Annotator_1 vs Annotator_2:
  Exact agreement:    0.377 (37.7%)
  Adjacent (+/-1):    0.713 (71.3%)
  Mean abs difference: 0.947
  Diff distribution:
0    113
1    101
2     75
3     11

Annotator_1 vs Annotator_3:
  Exact agreement:    0.420 (42.0%)
  Adjacent (+/-1):    0.853 (85.3%)
  Mean abs difference: 0.730
  Diff distribution:
0    126
1    130
2     43
3      1

Annotator_2 vs Annotator_3:
  Exact agreement:    0.750 (75.0%)
  Adjacent (+/-1):    0.920 (92.0%)
  Mean abs difference: 0.330
  Diff distribution:
0    225
1     51
2     24

PER-PERSONA BREAKDOWN: Annotator_1 vs Annotator_3
(the two non-straightlining annotators)

alpha (pα) — high health literacy  (n=100)
  Exact agreement:  0.740 (74.0%)
  Adjacent (+/-1):  0.950 (95.0%)
  Mean abs diff:    0.320
  A1 mean=4.68, A3 mean=5.00, A1-A3 systematic bias=-0.32

beta (pβ) — socioeconomic barrier  (n=100)
  Exact agreement:  0.340 (34.0%)
  Adjacent (+/-1